# 04 — Phase 4: Cross-Category Overlap (the headline result)

Jaccard overlap between the per-category head sets, reported against **two** nulls, because the
project treats both high and low overlap as informative and a single null can only bound one side.

| Null | Question it answers |
|---|---|
| I — hypergeometric chance floor | Is the overlap above what random k-subsets of 144 heads would give? |
| II — item-label permutation | Is it *below* what one shared mechanism would give? |

Null II is what makes a **low** overlap positive evidence for task-specific arbitration rather
than a bare null result.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2

In [2]:
from circuit_conflict import dataset as D, metrics as M, pipeline as PL
import numpy as np, pandas as pd, json

effects = {c: np.load(PL.RESULTS / "phase1" / f"effects_{c}.npy") for c in ["A", "B", "C"]}
stats_by_cat = {c: pd.read_csv(PL.RESULTS / "phase3" / f"head_stats_{c}.csv") for c in effects}
topk_sets = {c: M.top_k_head_set(s, k=10) for c, s in stats_by_cat.items()}

## The chance floor is not zero

In [3]:
null = M.jaccard_null_uniform(10, 10, n_heads=144)
print(f"  expected overlap by chance : {null['expected_jaccard']:.4f}")
print(f"  need >= {null['min_intersection_p05']:.0f} shared heads "
      f"(J >= {null['min_jaccard_p05']:.3f}) to clear p < 0.05")

  expected overlap by chance : 0.0360
  need >= 3 shared heads (J >= 0.176) to clear p < 0.05


## Observed overlap with both nulls, plus a bootstrap CI

In [4]:
selector = lambda e: M.top_k_head_set(M.paired_head_stats(e), k=10)
table = M.compile_cross_category_table(topk_sets, effects, selector,
                                       n_perm=200, n_boot=200, seed=0)
table.to_csv(PL.RESULTS / "phase4" / "cross_category_topk.csv", index=False)
table.round(4)

,cat_a,cat_b,n_a,n_b,intersection,jaccard,overlap_coefficient,ci_lo,ci_hi,expected_jaccard_chance,p_upper_vs_chance,perm_null_median,p_lower_vs_shared
0,A,B,10,10,3,0.1765,0.3,0.1111,0.3333,0.036,0.0226,0.6667,0.000
1,A,C,10,10,5,0.3333,0.5,0.2482,0.3333,0.036,0.0001,0.6026,0.000
2,B,C,10,10,4,0.2500,0.4,0.1765,0.3357,0.036,0.0021,0.6667,0.005


## Robustness

A conclusion that only holds at one choice of *k* is not a conclusion, and Spearman rho on the
full 144-head vectors avoids thresholding entirely.

In [5]:
ovk = M.overlap_vs_k(stats_by_cat)
ovk.to_csv(PL.RESULTS / "phase4" / "overlap_vs_k.csv", index=False)
print(ovk.pivot_table(index="k", columns=["cat_a", "cat_b"], values="jaccard").round(3))

scores = {c: s.pivot(index="layer", columns="head_idx", values="median_effect").values
          for c, s in stats_by_cat.items()}
rho = M.rank_correlation_across_categories(scores)
rho.to_csv(PL.RESULTS / "phase4" / "rank_correlation.csv", index=False)
rho.round(4)

cat_a      A             B
cat_b      B      C      C
k                         
5      0.250  0.250  0.250
10     0.176  0.333  0.250
20     0.290  0.290  0.250
30     0.364  0.304  0.395


,cat_a,cat_b,spearman_rho,p_value
0,A,B,0.4537,0.0
1,A,C,0.3924,0.0
2,B,C,0.5540,0.0
